# V2 original + split agrupado provisional — ASL

Nueva corrida de la arquitectura V2 original, confirmada en GitHub.
Se conserva ASL, crop RGB/depth 224×224, fusión por concatenación, dos Cross-Attention,
MLP compartido y cabezales de proporciones/masa total. NO incorpora cambios de V3–V6.

Datos: paquete agrupado aceptado de V3, sin volver a generar splits/vocabulario.
Resultados exclusivamente en **Drive/v2_grouped_asl/**. V2 histórica y V3–V6 quedan intactas.
No cargar checkpoints históricos: cambian entrenamiento y seis claves de vocabulario.
Todos los RUN están apagados. Usar sesión Colab nueva y GPU.


## 1. Montar Drive

Mantener el acceso directo en Mi unidad si la carpeta es compartida, o adaptar DRIVE en sección 3.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, hashlib
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")


## 2. Descargar la rama del control agrupado

Publicar primero v2-grouped-asl. Usa arquitectura/configuración V2 original con infraestructura compatible para el split nuevo y protección de resultados. No modifica la rama v2-multiscale-depth.


In [ ]:
URL = "https://github.com/marcogloazzo/IngredientMassEstimation.git"
BRANCH = "v2-grouped-asl"
PRIVATE_REPO = False
REPO = Path("/content/nutrition-code-v2-grouped")

def git_private(command):
    environment = os.environ.copy()
    try:
        if PRIVATE_REPO:
            from google.colab import userdata
            import base64
            token = userdata.get("GITHUB_TOKEN")
            authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            environment.update({
                "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {authorization}",
            })
            del token, authorization
        result = subprocess.run(command, env=environment, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Git falló: revisá URL, rama publicada y permisos del token. No se imprime la credencial.")
        return result.stdout.strip()
    finally:
        environment.clear()

if (REPO / ".git").is_dir():
    current = git_private(["git", "-C", str(REPO), "branch", "--show-current"])
    assert current == BRANCH, f"Esta carpeta tiene otra rama: {current}. Usá una carpeta nueva."
    git_private(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH])
else:
    git_private(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)])

PROJECT = next((p for p in (REPO, REPO / "IngredientMassEstimation")
                if (p / "config/query2label_100_v2_asl_grouped.json").is_file()), None)
assert PROJECT is not None, "Falta publicar el control V2 agrupado."
os.chdir(PROJECT)
print("Proyecto:", PROJECT)
print("Rama:", git_private(["git", "-C", str(REPO), "branch", "--show-current"]))

print("Commit:", git_private(["git", "-C", str(REPO), "rev-parse", "HEAD"]))


## 3. Dependencias y rutas exclusivas

No requiere CLIP. La V2 original mantiene ASL y la geometría 230→crop224. La media train de la pérdida total se recalcula con los registros nuevos.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-v2.txt", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
SOURCE = PROJECT / "src"
assert (SOURCE / "ingredient_mass").is_dir(), "Falta código del proyecto."
sys.path.insert(0, str(SOURCE))
import importlib.util
helper = SOURCE / "ingredient_mass/notebook_reports.py"
assert helper.is_file(), "Falta publicar notebook_reports.py."
spec = importlib.util.spec_from_file_location("v2_grouped_native_reports", helper)
reports = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reports)
show_saved_report = reports.show_saved_report
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DRIVE = Path("/content/drive/MyDrive/IngredientMassEstimation")
HISTORICAL_V2 = DRIVE / "v2"  # SOLO LECTURA
CONTROL = DRIVE / "v2_grouped_asl"
EXPERIMENT_TAG = "run_01"
REPORT_TAG = "eval_01"
assert all("/" not in tag and "\\" not in tag and tag not in ("", ".", "..") for tag in (EXPERIMENT_TAG, REPORT_TAG))
CONFIG = PROJECT / "config/query2label_100_v2_asl_grouped.json"
cfg = json.loads(CONFIG.read_text())
assert cfg["presence_loss"] == "asl" and cfg["experiment_namespace"] == "v2_grouped_asl"
assert cfg["depth"]["geometry"] == "rgb_crop_230_224_v1"
RUN = CONTROL / "runs" / f"query2label_100_v2_asl_{EXPERIMENT_TAG}"
VALIDATION = CONTROL / "reports" / f"{EXPERIMENT_TAG}_{REPORT_TAG}" / "validation"
TEST = VALIDATION.parent / "test"
CHECKPOINT = VALIDATION / "calibrated.pt"
LABELS_PARENT = Path("/content/v2_grouped_prepared")
PROCESSED = LABELS_PARENT / "processed_100_grouped_provisional"
DATA_PARENT = Path("/content/nutrition5k_v2_grouped")
DEPTH_REPO = Path("/content/depth-anything-v2-grouped")
DEPTH_WEIGHTS = CONTROL / "weights/depth_anything_v2_vits.pth"
DEPTH_CACHE = Path("/content/depth_v2_grouped_224")
DEPTH_ARCHIVE = CONTROL / "depth_v2_grouped_224.zip"
HISTORICAL_DEPTH_ARCHIVE = HISTORICAL_V2 / "depth_v2_vits.zip"
BASELINES = {"v3": DRIVE / "v3/reports/run_01/validation",
             "v4": DRIVE / "v4/reports/split_only_run_01_eval_01/validation",
             "v5": DRIVE / "v5/reports/run_01_eval_01/validation",
             "v6": DRIVE / "v6/reports/run_01_eval_01/validation"}

def run_stream(arguments):
    command = [str(arg) for arg in arguments]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

def new_output(path):
    assert path.resolve().is_relative_to(CONTROL.resolve()), f"Salida fuera del control nuevo: {path}"
    assert not path.exists() or not any(path.iterdir()), f"Ya tiene resultados: cambiar tag o usar resume, no sobrescribir {path}"

print("Device:", DEVICE)
print("V2 original: ASL, RGB/depth224, encoder E2–E4 tras warmup.")
print("Resultados nuevos:", CONTROL)
print("V2 histórica SOLO LECTURA:", HISTORICAL_V2)


## 4. RGB y etiquetas agrupadas originales

Reutiliza nutrition5k_rgb.zip y v3/data/processed_100_grouped_provisional.zip. No ejecutar prepare_nutrition5k.py ni crear otra validación. Verificar hashes.


In [ ]:
from pathlib import PurePosixPath
import stat

def extract_zip_safe(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        checked = []
        for info in source.infolist():
            name = info.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if relative.is_absolute() or not relative.parts or ":" in relative.parts[0]:
                raise ValueError(f"Nombre inseguro en ZIP: {name}")
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Ruta insegura en ZIP: {name}")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"No se aceptan enlaces en ZIP: {name}")
            checked.append((info, target, name.endswith("/")))
        for info, target, is_directory in checked:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(info) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)


def dataset_roots():
    if not DATA_PARENT.is_dir():
        return []
    return sorted({folder.parent for folder in DATA_PARENT.rglob("metadata")
                   if folder.is_dir()
                   and (folder / "dish_metadata_cafe1.csv").is_file()
                   and (folder / "ingredients_metadata.csv").is_file()
                   and (folder.parent / "dish_ids").is_dir()
                   and (folder.parent / "imagery/realsense_overhead").is_dir()})

roots = dataset_roots()
if not roots:
    source = DRIVE / "nutrition5k_rgb.zip"
    assert source.is_file(), f"Falta: {source}"
    local = Path("/content/nutrition5k_rgb_v2_grouped.zip")
    shutil.copy2(source, local)
    extract_zip_safe(local, DATA_PARENT)
    roots = dataset_roots()
assert len(roots) == 1, f"Raíz RGB no única: {roots}"
DATA = roots[0]

labels_zip = DRIVE / "v3/data/processed_100_grouped_provisional.zip"
assert labels_zip.is_file(), f"Falta el paquete agrupado existente: {labels_zip}"
if not (PROCESSED / "records.jsonl").is_file():
    extract_zip_safe(labels_zip, LABELS_PARENT)
package = json.loads((LABELS_PARENT / "package_manifest.json").read_text())
for name, digest in package["file_sha256"].items():
    path = PROCESSED / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Etiquetas modificadas: {name}"
summary = json.loads((PROCESSED / "summary.json").read_text())
assert summary["mode"] == "provisional" and summary["official_test_preserved"]
print("Paquete:", PROCESSED)
print("Splits:", summary["samples"])
print("No regenerar etiquetas: misma huella para comparar con V3–V6.")


## 5. Código/pesos Depth Anything congelado

Reutiliza los pesos Small existentes, copiándolos al control nuevo. No escribe en v2/v3 ni entrena el estimador.


In [ ]:
import urllib.request
depth_cfg = cfg["depth"]
if not (DEPTH_REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "https://github.com/DepthAnything/Depth-Anything-V2.git", DEPTH_REPO], check=True)
    subprocess.run(["git", "-C", DEPTH_REPO, "checkout", depth_cfg["repo_revision"]], check=True)
assert subprocess.check_output(["git", "-C", DEPTH_REPO, "rev-parse", "HEAD"], text=True).strip() == depth_cfg["repo_revision"]
DEPTH_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not DEPTH_WEIGHTS.is_file():
    previous = next((p for p in (HISTORICAL_V2 / "weights/depth_anything_v2_vits.pth",
                                DRIVE / "v3/weights/depth_anything_v2_vits.pth") if p.is_file()), None)
    if previous:
        shutil.copy2(previous, DEPTH_WEIGHTS)
    else:
        urllib.request.urlretrieve("https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth",
                                   DEPTH_WEIGHTS)
print("Pesos del control nuevo:", DEPTH_WEIGHTS)


## 6. Cache V2 224×224 y controles — NO entrenamiento

**No usar mapas V3–V6 de 320×320 ni redimensionarlos a 224.**
Se intenta restaurar el ZIP de esta corrida; si falta, copia el ZIP histórico V2 en modo lectura.
El cache es independiente de split/vocabulario: se reutiliza únicamente si coinciden
geometría, RGB, pesos y todos los IDs requeridos. Si faltan mapas, RUN_DEPTH_PREPARATION=True
los genera/completa en el cache nuevo. El original nunca se modifica.

RUN_DATA_CHECK=True verifica el control antes de entrenar. La generación de mapas es inferencia congelada, no training.


In [ ]:
RUN_DEPTH_PREPARATION = False
RUN_DATA_CHECK = False
if not (DEPTH_CACHE / "manifest.json").is_file():
    source_archive = DEPTH_ARCHIVE if DEPTH_ARCHIVE.is_file() else HISTORICAL_DEPTH_ARCHIVE
    if source_archive.is_file():
        local = Path("/content/depth_v2_grouped_restore.zip")
        shutil.copy2(source_archive, local)
        staging = Path("/content/v2_grouped_depth_restore")
        assert not staging.exists(), "La restauración temporal ya existe: usar otro staging si el cache quedó incompleto."
        extract_zip_safe(local, staging)
        roots = []
        for manifest_path in staging.rglob("manifest.json"):
            details = json.loads(manifest_path.read_text())
            if details.get("depth_config") == depth_cfg:
                roots.append(manifest_path.parent)
        assert len(roots) == 1, f"No se encontró raíz única del cache224: {roots}"
        assert roots[0].resolve().is_relative_to(staging.resolve())
        assert not DEPTH_CACHE.exists(), "Cache local parcial: elegir una carpeta local nueva, no sobrescribir."
        shutil.copytree(roots[0], DEPTH_CACHE)
        print("Restaurado en carpeta NUEVA desde:", source_archive)
    else:
        print("No hay ZIP224 disponible: habilitar RUN_DEPTH_PREPARATION.")

if RUN_DEPTH_PREPARATION:
    run_stream([sys.executable, "-u", "scripts/prepare_depth.py", "--config", CONFIG,
                "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
                "--depth-anything-repo", DEPTH_REPO, "--depth-checkpoint", DEPTH_WEIGHTS, "--device", DEVICE])
if (DEPTH_CACHE / "manifest.json").is_file():
    manifest = json.loads((DEPTH_CACHE / "manifest.json").read_text())
    assert manifest["depth_config"] == depth_cfg, "Cache de otra geometría/config."
    assert hashlib.sha256(DEPTH_WEIGHTS.read_bytes()).hexdigest() == manifest["checkpoint_sha256"], "Cache y pesos depth no coinciden."
    run_stream([sys.executable, "-u", "scripts/prepare_depth.py", "--config", CONFIG,
                "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
                "--check-cache", "--device", DEVICE])
    if not DEPTH_ARCHIVE.exists():
        local_zip = Path(shutil.make_archive("/content/depth_v2_grouped_224_export", "zip",
                                            root_dir=DEPTH_CACHE.parent, base_dir=DEPTH_CACHE.name))
        DEPTH_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_zip, DEPTH_ARCHIVE)
        print("Cache verificado guardado en Drive nuevo:", DEPTH_ARCHIVE)
    if RUN_DATA_CHECK:
        run_stream([sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG,
                    "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
                    "--device", DEVICE, "--check-data"])
else:
    print("Falta cache224: no se puede entrenar todavía.")


## 7. Reentrenamiento V2 agrupado — apagado

Primera corrida MODE="new": inicialización ImageNet, NO checkpoint V2 histórico.
ASL, batch4×accum8, máximo100 épocas, warmup3, paciencia8 por L1 de ingredientes;
misma arquitectura y parámetros de configuración que V2 original salvo datos/protecciones.
El paquete agrupado cambia el vocabulario/perfiles respecto a V2 histórica; no reutilizar sus cabezales.
Resultados y resume exclusivamente en v2_grouped_asl/.


In [ ]:
RUN_TRAINING = False
MODE = "new"
TOTAL_EPOCHS = 100
ALLOW_CPU_TRAINING = False
if RUN_TRAINING:
    assert DEVICE == "cuda" or ALLOW_CPU_TRAINING, "Elegí GPU en Colab."
    assert (DEPTH_CACHE / "manifest.json").is_file(), "Primero verificar cache224 en sección 6."
    assert MODE in ("new", "resume")
    args = [sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG,
            "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
            "--output-dir", RUN, "--torch-cache", DRIVE / "torch_cache",
            "--device", DEVICE, "--workers", "2", "--epochs", TOTAL_EPOCHS]
    if MODE == "resume":
        assert (RUN / "last.pt").is_file(), "Falta last.pt del NUEVO control agrupado."
        args += ["--resume", RUN / "last.pt"]
    else:
        new_output(RUN)
    run_stream(args)
else:
    print("Entrenamiento V2 agrupado desactivado.")


## 8. Validación/calibración y test apagados

REPORT_TAG nuevo para repetir. La calibración de presencia V2 no cambia gramos. Nunca evaluar/resumir el checkpoint V2 histórico con este nuevo vocabulario.


In [ ]:
RUN_CALIBRATION = False
RUN_TEST = False
shared = ["--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
          "--device", DEVICE, "--batch-size", "4", "--workers", "2"]
if RUN_CALIBRATION:
    assert (RUN / "best.pt").is_file(), "Falta best.pt del control agrupado."
    new_output(VALIDATION)
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", RUN / "best.pt",
                "--split", "validation", "--calibrate", "--output-dir", VALIDATION, *shared])
if RUN_TEST:
    assert CHECKPOINT.is_file(), "Primero calibrar esta V2 agrupada."
    new_output(TEST)
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", CHECKPOINT,
                "--split", "test", "--output-dir", TEST, *shared])
for split, folder in (("V2 agrupada · validación", VALIDATION), ("V2 agrupada · test", TEST)):
    figures = show_saved_report(folder / "metrics.json", "metrics", split)


## 9. Errores por ingrediente y comparación controlada — solo resultados guardados

V2 usa proporciones×masa total: no ejecutar el diagnóstico de cantidad condicional p×c de V3–V6. La tabla reconstruye los gramos de V2 desde su NPZ. Comparaciones con mismo split/vocabulario; V2 histórica solo sería descriptiva.


In [ ]:
RUN_COMPARISON = False
source = VALIDATION / "predictions.npz"
if source.is_file():
    import numpy as np
    vocabulary = json.loads((PROCESSED / "vocabulary.json").read_text())
    with np.load(source, allow_pickle=False) as saved:
        keys = saved["vocabulary_keys"].astype(str).tolist()
        assert keys == [entry["key"] for entry in vocabulary["entries"]]
        predicted = saved["predicted_shares"] * saved["predicted_total_mass"][:, None]
        truth = saved["masses"]
    rows = []
    for index, entry in enumerate(vocabulary["entries"]):
        present = truth[:, index] > 0
        error = np.abs(predicted[:, index] - truth[:, index])
        rows.append({"name": entry["name"], "support": int(present.sum()),
                     "mae_present": float(error[present].mean()) if present.any() else None,
                     "l1": float(error.mean()), "false_g": float((predicted[:, index] * ~present).mean())})
    rows.sort(key=lambda row: row["l1"], reverse=True)
    figure = reports._table("V2 agrupada · ingredientes de mayor error", rows[:15],
        [("name","Ingrediente"),("support","Soporte"),("mae_present","MAE presente g"),
         ("l1","L1 g/plato"),("false_g","Falsos g/plato")], width=14, first_width=.35)
else:
    print("Todavía no hay predicciones de validación guardadas.")
for label, previous in BASELINES.items():
    comparison_dir = CONTROL / "comparisons" / f"{label}_vs_v2_grouped_{EXPERIMENT_TAG}_{REPORT_TAG}"
    if RUN_COMPARISON:
        if not (previous / "metrics.json").is_file():
            print("Referencia todavía no disponible:", previous)
            continue
        new_output(comparison_dir)
        run_stream([sys.executable, "-u", "scripts/compare_image_experiments.py",
                    "--left", previous, "--right", VALIDATION, "--output-dir", comparison_dir])
    figures = show_saved_report(comparison_dir / "comparison.json", "comparison", f"{label} vs V2 agrupada")


## 10. Inferencia opcional con referencias y atención — apagada

Por defecto foto de validación. La referencia real no se pasa al modelo. Usar INFERENCE_TAG nuevo para preservar predicciones anteriores.


In [ ]:
RUN_INFERENCE = False
PHOTO_SOURCE = "validation"  # validation / test / personal
PLATE_INDEX = 0
INFERENCE_TAG = "photo_01"
if RUN_INFERENCE:
    assert CHECKPOINT.is_file()
    assert "/" not in INFERENCE_TAG and "\\" not in INFERENCE_TAG and INFERENCE_TAG not in ("", ".", "..")
    reference = None
    if PHOTO_SOURCE == "personal":
        PHOTO = DRIVE / "photos/IMG_9993.png"
    else:
        assert PHOTO_SOURCE in ("validation", "test")
        selected = [json.loads(line) for line in (PROCESSED / "records.jsonl").read_text().splitlines()
                    if json.loads(line)["split"] == PHOTO_SOURCE]
        assert 0 <= PLATE_INDEX < len(selected)
        reference = selected[PLATE_INDEX]
        PHOTO = DATA / reference["image"]
    assert PHOTO.is_file()
    prediction_dir = CONTROL / "predictions" / f"{EXPERIMENT_TAG}_{REPORT_TAG}_{INFERENCE_TAG}"
    new_output(prediction_dir)
    run_stream([sys.executable, "-u", "scripts/infer_image_model.py", "--checkpoint", CHECKPOINT,
                "--image", PHOTO, "--output-dir", prediction_dir, "--device", DEVICE,
                "--depth-anything-repo", DEPTH_REPO, "--depth-checkpoint", DEPTH_WEIGHTS,
                "--save-attention", "--top-k", "6"])
    saved = prediction_dir / PHOTO.stem
    result = json.loads((saved / "prediction.json").read_text())
    rows = [{"metric":"Masa total (g)","truth":reference["total_mass"] if reference else None,
             "predicted":result["predicted_total_mass_grams"]}]
    for key, unit in (("calories","kcal"),("fat","g"),("carbohydrates","g"),("protein","g")):
        rows.append({"metric":f"{key} ({unit})","truth":reference["nutrients"][key] if reference else None,
                     "predicted":result["nutrition"][key]})
    figure = reports._table("Referencia vs V2 agrupada",rows,[("metric","Métrica"),("truth","Real"),("predicted","Estimado")])
    figure = reports._table("Ingredientes estimados",result["ingredients"][:12],
        [("name","Ingrediente"),("probability","p"),("grams","Gramos"),("recognized","Reconocido")],width=12,first_width=.40)
    from PIL import Image
    import matplotlib.pyplot as plt
    images = [(saved / "intermediates/input_crop.png","RGB crop224"),
              (saved / "intermediates/estimated_relative_depth.png","Depth relativo224")]
    images += [(path,path.stem) for path in sorted((saved / "intermediates").glob("*_attention.png"))[:2]]
    fig, axes = plt.subplots(1,len(images),figsize=(4*len(images),4))
    for ax,(path,title) in zip(axes,images):
        if path.is_file():
            with Image.open(path) as img: ax.imshow(img.copy())
        ax.set_title(title); ax.axis("off")
    fig.tight_layout(); plt.show(); plt.close(fig)
else:
    print("Inferencia V2 agrupada desactivada.")


## Git y Drive: preservar todo

Git: nueva rama v2-grouped-asl, notebook/config/documento/pruebas del control y cambios de protección/UI.
No incluir notebooks anteriores ejecutados ni pesos/caches/datasets/resultados.
Referencia V2 original: commit 3106056b4a64c68d9358b35753f1a2d2af1f01dd de v2-multiscale-depth.
No hay commit/push automático ni cambio de rama realizado por este notebook.

Drive: conservar v2/ y v3–v6/. Reutilizar RGB y paquete agrupado de v3/.
El cache224 histórico puede COPIARSE y verificarse; jamás modificarlo o sustituirlo por mapas320.
Nuevo control/copia de pesos/cache/runs/reports/predictions en v2_grouped_asl/.
Para resume: solo last.pt de esa nueva carpeta. No resume ni warm-start desde V2 histórica.
